# Quasi-harmonic free energy in practice: MD, a thermostat you can defend, and two routes

This tutorial produces a vibrational entropy from a molecular dynamics trajectory and
turns it into `G - E_el`. The molecule is **acetone** (`dsgdb9nsd_000018`, 10 atoms),
which is small enough that every step here runs in minutes rather than hours.

It is the practical companion to `docs/branchB_workflow.md`. Where that document states
the protocol, this one runs it and shows the numbers coming out.

**What you will end up having measured, not assumed:**

1. that the estimator is exact on a case whose answer is known in closed form;
2. what the thermostat costs, against that closed form;
3. that two independent production routes -- ASE and OpenMM -- agree;
4. that an independent implementation of the superposition agrees (and that one of the two
   obvious candidates does *not*, for a reason worth knowing).

## 0. Setup

Branch B has two production routes for the same protocol -- ASE and OpenMM -- and **one
environment runs both**. That was not always true: `openmm-torch` pins the pytorch it was
compiled against, so the OpenMM stack used to need an environment of its own.

On 2026-09-05 that pin was accepted (pytorch 2.13.0 -> 2.12.1, numpy 2.4.6 -> 1.26.4) and
**the cost was measured rather than assumed**. On identical geometry, weights and frames,
energy, forces, every Hessian frequency, `T*S` on 2000 fixed frames and every
quasi-harmonic frequency all came back **bit-identical**
(`scripts/calibration/s0_B_stack_fingerprint.py`).

Take that fingerprint before and after any change to the environment files. A stack change
whose cost is unmeasured is a stack change nobody can defend -- and note what it *cannot*
tell you: it compares numbers, so an API that no longer exists produces nothing to
compare. `np.trapezoid` disappearing in numpy 1.26 was found by the test suite, not by the
fingerprint.

Start by asking this installation what it can do. The probes are real imports and real
executions, never `find_spec` -- a present-but-broken build must read as unavailable.


In [ ]:
import json, subprocess, sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
while not (ROOT / "openqha" / "__init__.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from openqha import capabilities, config, engine, qha, hessian, thermo

cfg = config.load()
SPECIES = "dsgdb9nsd_000018"
print("repo   ", ROOT)
print("engine ", engine.engine_name())
print("T      ", config.temperature(cfg), "K")
print()
print(capabilities.summary())


## 1. The estimator, on a case where the answer is known

Before trusting a number from a trajectory, check the machinery on something whose answer
is written down. For a harmonic well the classical canonical distribution satisfies

$$\langle q_k^2 \rangle = \frac{k_B T}{\omega_k^2}$$

**exactly**, so a correctly sampled trajectory must return the Hessian's own frequencies.
`qha.harmonic_limit_check` samples that distribution analytically and pushes it through
the production chain.

Two numbers come back and they answer different questions. `exact_covariance_TS_error`
uses `lambda_k = k_B T / omega_k^2` directly with no sampling at all -- it isolates the
algebra and should sit at machine precision. `sampled_TS_error` runs the fitted-covariance
chain over finite frames and carries the estimator's statistical error; it is what
acceptance criterion 3 is written on.

In [ ]:
from ase.io import read
from ase.optimize import BFGS

atoms = read(str(config.qm9_xyz(SPECIES, cfg)))
calc, name, prov = engine.calculator(device="cpu")
atoms.calc = calc
BFGS(atoms, logfile=None).run(fmax=1e-4, steps=2000)

H, asymmetry = hessian.hessian(atoms, calc, mode="analytic")
print("Hessian max|H - H^T| = {:.3e} eV/A^2".format(asymmetry))

check = qha.harmonic_limit_check(H, atoms.get_masses(), atoms.get_positions(),
                                 temperature_K=298.15, n_frames=100000)
print("closed-form T*S           {:.6f} kcal/mol".format(
    check["reference"]["closed_form_TS_kcal"]))
print("exact-covariance error    {:.3e} kcal/mol   <- the algebra".format(check["exact_covariance_TS_error_kcal"]))
print("sampled error             {:+.4f} kcal/mol   <- the estimator (budget 0.02)".format(
    check["sampled_TS_error_kcal"]))
print("RMS frequency error       {:.4f} cm^-1".format(check["rms_frequency_error_cm_inv"]))

## 2. What the thermostat costs

The same harmonic surface turns "which thermostat?" from a comparison between thermostats
-- which can only ever show that they disagree -- into a measurement of each one's **bias
against truth**.

Run the calibration. It is deliberately loaded with two variants that *must* be rejected:
`berendsen`, which has no canonical stationary distribution, and `nose_hoover_chain_1`,
which is plain Nosé-Hoover -- the harmonic oscillator is the textbook case where it is not
ergodic. If they pass, suspect the script rather than believing the table.

In [ ]:
cmd = [sys.executable, str(ROOT / "scripts/calibration/s0_B_thermostat_choice.py"),
       "--steps", "200000", "--seeds", "2", "--processes", "8",
       "--variants", "exact_sampling", "langevin_1.0",
       "nh_tdamp20_c3_l1", "nose_hoover_chain_3", "nh_tdamp419_c3_l1",
       "berendsen", "nose_hoover_chain_1",
       "--out", "/tmp/t01b_thermostat"]
print(" ".join(cmd), "\n")
print(subprocess.run(cmd, capture_output=True, text=True).stdout[-2500:])

### How to read that table

`kin dof` is `2<KE>/kT`, read off the sample rather than assumed -- it should be `3N = 30`.
`conf.var ratio` is the variance of the potential energy divided by its canonical value;
it should be 1.

The row that matters is not the error alone. **Nosé-Hoover holds the kinetic temperature
perfectly correct at every coupling time while the configurational distribution
collapses**, and quasi-harmonic analysis reads only the configurational one. At a 419 fs
coupling the softest mode comes back near 229 cm⁻¹ against a true 79.7.

A temperature check cannot see this. That is why `qha.assert_trajectory_identity` refuses
a coupling time above 20 fs instead of trusting the reported temperature.

**20 fs is not this repository's invention.** It is `openmmtools`' documented default
(`collision_frequency = 50/ps`), and the same 50 ps⁻¹ appears in the QHA reference paper
this branch cites.

## 3. Route one: ASE

Short runs so the notebook finishes. Production lengths are set by the saturation curve,
not by a constant -- see §5.

In [ ]:
cmd = [sys.executable, str(ROOT / "scripts/production/s0_B_qha_trajectory.py"),
       "--species", SPECIES, "--tag", "t01b_ase", "--seeds", "2",
       "--equil-ps", "0.5", "--prod-ps", "3"]
print(" ".join(cmd), "\n")
out = subprocess.run(cmd, capture_output=True, text=True).stdout
print("\n".join(l for l in out.splitlines() if l.strip())[-1800:])

Notice the protocol line: it names the thermostat actually in force and its coupling time.
It used to print "friction", which is a Langevin parameter that a Nosé-Hoover run does not
have -- a banner that names a parameter the run does not possess is how a product ends up
recording a request rather than what ran.

## 4. Route two: OpenMM

The same protocol, a different implementation of it -- in the same interpreter. The driver
**refuses to run** unless its force reproduces the ASE calculator, so the first thing it
prints is that check.

MACE gets into OpenMM through `openqha/openmm_mace.py`, which builds the interaction graph
as the **complete graph** rather than doing a neighbour search. For 10-19 atoms that is
exact -- pairs beyond `r_max` are multiplied by a cutoff function that is exactly zero --
and it means this route cannot have an origin-anchored neighbour list, because there is no
box to anchor.

If `capabilities.summary()` above reported the OpenMM core as missing, this cell will say
so and the rest of the notebook still runs on the ASE route alone.


In [ ]:
cmd = [sys.executable, str(ROOT / "scripts/production/s0_B_qha_trajectory_openmm.py"),
       "--species", SPECIES, "--tag", "t01b_omm", "--seeds", "2",
       "--equil-ps", "0.5", "--prod-ps", "3"]
if all(capabilities.available(c) for c in capabilities.CORE):
    print(" ".join(cmd), "\n")
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    print("\n".join(l for l in out.splitlines() if l.strip())[-1800:])
else:
    missing = [c for c in capabilities.CORE if not capabilities.available(c)]
    print("SKIPPED: branch B's OpenMM core is missing {}.".format(", ".join(missing)))
    print("conda env update -f environment.yml --prune")


## 5. Analysis, and the criterion that is supposed to fail

One analysis reads either route. It prints every acceptance criterion and returns non-zero
if any fails.

Expect **criterion 1 to fail** on a 3 ps run, and expect it to fail on a 25 ps run too.
Andricioaei and Karplus needed about 4 ns where 200 ps was not converged. A saturation
criterion that passed at 3 ps would be measuring nothing -- and a 0.4 ps run once did pass
it, which is why the analysis now refuses to score criteria 1 and 5 below 20 ps.

In [ ]:
for tag in ("t01b_ase", "t01b_omm"):
    cmd = [sys.executable, str(ROOT / "scripts/production/s0_B_qha_analyse.py"),
           "--species", SPECIES, "--tag", tag]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    print("=" * 88)
    print("##", tag)
    print("=" * 88)
    print("\n".join(l for l in out.splitlines()
                     if l.startswith(("basin", "[PASS]", "[FAIL]", "   ")))[:2200])
    print()

## 6. Do the two routes agree?

This is the comparison the two-route design exists for. The two use different integrators,
different chain-propagation orders, different neighbour-list constructions and different
MACE installs. If they agree within the seed-to-seed noise floor, that is evidence about
the protocol. If they do not, it is a measurement, not a mystery.

In [ ]:
def read_TS(tag):
    root = Path.home() / "runs" / "openQHA" / "qha" / tag / SPECIES / "basin00"
    out = []
    for seed_dir in sorted(root.glob("seed*")):
        frames = np.load(seed_dir / "frames.npy")
        meta = json.loads((seed_dir / "meta.json").read_text())
        rec = qha.analyse(frames, meta["masses_amu"], meta["temperature_K"], meta=meta)
        out.append(rec["entropy"]["TS_QH_kcal"])
    return np.array(out)

ase_TS, omm_TS = read_TS("t01b_ase"), read_TS("t01b_omm")
print("ASE route     T*S = {}  mean {:.4f}".format(np.round(ase_TS, 4), ase_TS.mean()))
print("OpenMM route  T*S = {}  mean {:.4f}".format(np.round(omm_TS, 4), omm_TS.mean()))
print("difference of means           {:+.4f} kcal/mol".format(omm_TS.mean() - ase_TS.mean()))
print("seed-to-seed spread, ASE      {:.4f}".format(ase_TS.std(ddof=1)))
print("seed-to-seed spread, OpenMM   {:.4f}".format(omm_TS.std(ddof=1)))
print()
print("At 3 ps the spread is large; read this as a smoke test of the comparison, not")
print("as a result. The production comparison is at the length criterion 1 asks for.")

## 7. Independent superposition -- and why one library is not enough

The superposition is not preprocessing; it is half the estimator. `openqha/qha.py` fits
mass-weighted and iterates the reference to the mean of the fitted frames. Two widely used
libraries offer an independent implementation.

They are **not equivalent**, and the difference is not subtle.

In [ ]:
from openqha import mdtraj_io, gmx_io

seed_dir = Path.home() / "runs" / "openQHA" / "qha" / "t01b_ase" / SPECIES / "basin00" / "seed00"
frames = np.load(seed_dir / "frames.npy")
meta = json.loads((seed_dir / "meta.json").read_text())

rec = mdtraj_io.cross_check(frames, meta["masses_amu"], meta["symbols"],
                            meta["temperature_K"])
print("available:", rec["available"])
print("ours          T*S = {:.4f}   (mass-weighted, iterated)".format(
    rec["ours"]["TS_QH_kcal"]))
for c in rec["comparisons"]:
    if "TS_QH_kcal" in c:
        print("{:<13} T*S = {:.4f}   diff {:+.4f} kcal/mol   mass-weighted={}".format(
            c["library"], c["TS_QH_kcal"], c["TS_difference_kcal"], c["mass_weighted"]))
    else:
        print("{:<13} {}".format(c["library"], c.get("skipped") or c.get("failed")))

On a real 25 ps trajectory this comparison decomposes the whole disagreement:

| implementation | mass-weighted | reference | `T*S` | difference |
|---|---|---|---|---|
| `openqha.qha` (ours) | yes | iterated mean | 5.799829 | — |
| **MDAnalysis, iterated** | yes | iterated mean | 5.799829 | **+8.020e-09** |
| MDAnalysis, as it comes | yes | one frame | 5.828531 | +2.870e-02 |
| `gmx covar -mwa` | yes | our mean structure | — | +1.27e-04 |
| `mdtraj` | **no** | one frame | 8.207780 | **+2.407951** |

Three separate things, separated:

* **not mass-weighting costs 2.41 kcal/mol** — 2.4x the whole stage-0 target, from the
  superposition alone;
* **not iterating the reference costs 0.029** — the bias `superimpose` warns about, now a
  number;
* **the implementation itself costs 8.0e-09** — our Kabsch/SVD against MDAnalysis'
  Theobald QCP quaternion, weighting and protocol matched. The residual is float32:
  MDAnalysis stores positions in single precision.

That third line is the one worth having: two independent algorithms, written by different
people, agreeing on this branch's estimator to nine significant figures.

mdtraj also printed, from its C extension,

```
mdtraj/rmsd/src/theobald_rmsd.cpp UNCONVERGED ROTATION MATRIX. RETURNING IDENTITY=300
```

— for 300 of 3125 frames its float32 solver returned the *identity* rotation instead of a
fit, silently as far as the Python API is concerned. Worth knowing before trusting any
RMSD-based analysis of these trajectories.


## 8. From entropy to `G - E_el`

The assembly calls the same `thermo.g_minus_eel` that branch C's Hessian route calls, with
the same explicit symmetry number and electronic degeneracy. That is what makes acceptance
criterion 7 true by construction: the translational, rotational and electronic terms are
bit-for-bit identical between the two routes, so the difference between them is the
vibrational treatment and nothing else.

The symmetry number is **declared, never derived** -- `pymsym` returns `C1` for a structure
0.005 Å off ideal symmetry, and these basins are further off than that.

In [ ]:
rec = qha.analyse(frames, meta["masses_amu"], meta["temperature_K"], meta=meta)
lam = np.asarray(rec["spectrum"]["eigenvalues_amu_A2"], dtype=float)
g = qha.g_minus_eel(np.asarray(meta["masses_amu"]), frames.mean(axis=0), lam,
                    symmetry_number=2, degeneracy=1,
                    temperature_K=meta["temperature_K"])
for k in ("S_trans_kcal_per_K", "S_rot_kcal_per_K", "S_vib_kcal_per_K",
          "G_minus_Eel_kcal"):
    if k in g:
        print("{:<24} {:.6f}".format(k, g[k]))

## What to take away

* **The thermostat is a measured choice with a stated cost.** Nosé-Hoover at 20 fs costs
  about −0.13 kcal/mol against the closed form; Langevin at 1/ps costs +0.004. Branch B
  runs Nosé-Hoover on instruction, and the cost is recorded rather than hidden.
* **A correct kinetic temperature proves nothing here.** Every failing Nosé-Hoover setting
  reported 298 K perfectly.
* **Criterion 1 failing at 25 ps is the system working**, not a bug to route around.
* **Two implementations of anything are worth one implementation and a lot of prose.**
  That applies to the production route, to the superposition, and to the thermostat.

Next: `docs/branchB_workflow.md` for the operational detail and the reasoning behind each
choice.